# Transcrição de Vídeos do YouTube

Este notebook automatiza o processo de verificação, download e transcrição de áudios de vídeos do YouTube.

**Pipeline**: Verificação de disponibilidade → Download de áudios (yt-dlp) → Transcrição (Whisper) → Compactação e limpeza

In [1]:
import pandas as pd
import warnings
import requests
import os
import subprocess
import whisper
import shutil
from tqdm.notebook import tqdm
import multiprocessing as mp
from itertools import cycle
import torch

warnings.filterwarnings('ignore')

In [2]:
# Para o meu caso pessoal, usando o meu notebook
project_base_path = 'G:/Meu Drive/trabalho'

required_folders_relative = [
    "cleaned_data",
    "filtrated_data_log",
    "download_log",
    "audios",
    "transcriptions"
]

for folder_name in required_folders_relative:
    full_folder_path = os.path.join(project_base_path, folder_name)
    if not os.path.exists(full_folder_path):
        os.makedirs(full_folder_path)
        print(f"Pasta criada: {full_folder_path}")

In [3]:
# df = pd.read_csv('G:/Meu Drive/trabalho/cleaned_data/final_comments_with_sentiment.csv')
df = pd.read_csv(os.path.join(project_base_path, 'cleaned_data', 'final_comments_with_sentiment.csv'))

In [4]:
df['video_id'].nunique()

8155

---
## 1. Verificação de Disponibilidade de Vídeos

Esta seção verifica quais vídeos do dataset ainda estão disponíveis no YouTube

O primeiro passo que precisamos fazer é filtrar os vídeos que estão no nosso dataset para sabermos quais ainda estão disponíveis no Youtube.

In [5]:
def check_video_availability(video_id):
    url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
    response = requests.get(url)
    
    return response.status_code == 200

In [6]:
def save_progress(valid_videos, invalid_videos):
    """Salva os vídeos válidos e inválidos em arquivos CSV separados."""
    # valid_videos.to_csv("G:/Meu Drive/trabalho/filtrated_data_log/valid_videos.csv", index=False)
    valid_videos.to_csv(os.path.join(project_base_path, 'filtrated_data_log', 'valid_videos.csv'), index=False)
    # invalid_videos.to_csv("G:/Meu Drive/trabalho/filtrated_data_log/invalid_videos.csv", index=False)
    invalid_videos.to_csv(os.path.join(project_base_path, 'filtrated_data_log', 'invalid_videos.csv'), index=False)

In [7]:
def load_progress():
    """Carrega o progresso de vídeos válidos e inválidos (se houver)."""
    try:
        # valid_videos = pd.read_csv("G:/Meu Drive/trabalho/filtrated_data_log/valid_videos.csv")
        valid_videos = pd.read_csv(os.path.join(project_base_path, 'filtrated_data_log', 'valid_videos.csv'))
    except FileNotFoundError:
        valid_videos = pd.DataFrame(columns=["video_id"])
    
    try:
        # invalid_videos = pd.read_csv("G:/Meu Drive/trabalho/filtrated_data_log/invalid_videos.csv")
        invalid_videos = pd.read_csv(os.path.join(project_base_path, 'filtrated_data_log', 'invalid_videos.csv'))
    except FileNotFoundError:
        invalid_videos = pd.DataFrame(columns=["video_id"])
    
    return valid_videos, invalid_videos

In [8]:
valid_videos, invalid_videos = load_progress()

In [9]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Verificando disponibilidade de vídeos"):
    if video_id not in valid_videos["video_id"].values and video_id not in invalid_videos["video_id"].values:
        status = check_video_availability(video_id)
        
        if status is not None:
            new_row = pd.DataFrame({"video_id": [video_id]})
            if status:
                valid_videos = pd.concat([valid_videos, new_row], ignore_index=True)
                #print(f"O vídeo {video_id} está disponível.")
            else:
                invalid_videos = pd.concat([invalid_videos, new_row], ignore_index=True)
               # print(f"O vídeo {video_id} foi removido ou está indisponível.")
        
        save_progress(valid_videos, invalid_videos)

Verificando disponibilidade de vídeos:   0%|          | 0/8155 [00:00<?, ?it/s]

---
## 2. Filtragem do Dataset

Filtra o dataset original mantendo apenas os comentários dos vídeos que ainda estão disponíveis. Isso garante que apenas conteúdo acessível seja processado nas etapas seguintes.

In [10]:
# df_dados = pd.read_csv('../raw_data/videos_info_clean.csv')
# df_filtro = pd.read_csv('G:/Meu Drive/trabalho/filtrated_data_log/valid_videos.csv')
df_filtro = pd.read_csv(os.path.join(project_base_path, 'filtrated_data_log', 'valid_videos.csv'))

In [11]:
df['video_id'] = df['video_id'].astype(str)
df_filtro['video_id'] = df_filtro['video_id'].astype(str)

In [12]:
dados_filtrados = df[df['video_id'].isin(df_filtro['video_id'])]

In [13]:
# dados_filtrados.to_csv('G:/Meu Drive/trabalho/filtrated_data_log/videos_filtrados.csv', index=False)
# dados_filtrados.to_csv(os.path.join(project_base_path, 'filtrated_data_log', 'videos_filtrados.csv'), index=False)
dados_filtrados.head()

,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,is_reply,parent_id,channel_id,language,language_langdetect,number_comments_videos,number_comments_users,number_videos_users,comment_length,sentiment
0,g3YDGLa2pWg,UgzP9hlUh9w_IWQksvN4AaABAg,@dr.gabrielfeitosa,https://yt3.ggpht.com/yp_y0N8quVmXxwhO3S1rQqlX...,http://www.youtube.com/@dr.gabrielfeitosa,UCYgV9EbxVSNdDHBr0GWLCmQ,Segue uma playlist sobre colaterais de hormôni...,2024-12-31 23:07:02+00:00,2024-12-31 23:07:02+00:00,1,0,root,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt,93,750,129,138,Neutral
1,g3YDGLa2pWg,UgzP9hlUh9w_IWQksvN4AaABAg.ACjBEF6NnUrACpvH__rSZ6,@Fabio_Britto,https://yt3.ggpht.com/ytc/AIdro_lbzoKmUeO5NasE...,http://www.youtube.com/@Fabio_Britto,UCaHIZ0V18SSmxIPQbKRyf4w,"Por favor. Não fumo, não bebo a 2 anos, tinha ...",2025-01-03 13:53:38+00:00,2025-01-03 13:53:38+00:00,0,1,UgzP9hlUh9w_IWQksvN4AaABAg,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt,93,2,129,153,Neutral
2,g3YDGLa2pWg,UgzNj3A5ReT4piuNYEN4AaABAg,@debyomental,https://yt3.ggpht.com/vuK55ebWZ0zCTt7q1tCiFAS6...,http://www.youtube.com/@debyomental,UCQsZB-Cd3kpnCjxbslnm1kg,Obrigado dr.gabriel! Eu sempre tive o sonho de...,2025-01-08 19:55:43+00:00,2025-01-08 19:55:43+00:00,0,0,root,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt,93,3,129,346,Positive
3,g3YDGLa2pWg,UgzNj3A5ReT4piuNYEN4AaABAg.AD2RgwaVf-aAD9OQUOJ1MM,@diegosilva3310,https://yt3.ggpht.com/ytc/AIdro_mZTUGWmVKXIHr5...,http://www.youtube.com/@diegosilva3310,UCu4uK6PpfnOegglK4TpStwA,"@@debyomental afere a pressao mano, monitora c...",2025-01-11 12:41:48+00:00,2025-01-11 12:41:48+00:00,0,1,UgzNj3A5ReT4piuNYEN4AaABAg,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt,93,18,129,59,Neutral
4,g3YDGLa2pWg,UgyN15Q5w3bYXx73vuV4AaABAg,@ancientknight4653,https://yt3.ggpht.com/ytc/AIdro_nAn5L24RfZZyrv...,http://www.youtube.com/@ancientknight4653,UCZ6A0MP48-hUXNTf9EQGBqQ,Muito informativo. Desejo crescimento para o c...,2025-01-06 01:32:15+00:00,2025-01-06 01:32:15+00:00,0,0,root,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt,93,2,129,106,Negative


---
## 3. Download de Áudios

Baixa os áudios dos vídeos disponíveis usando **yt-dlp**:
- **Formato**: MP3 (melhor qualidade de áudio disponível)
- **Cookies**: Utiliza arquivo de cookies para contornar restrições
- **Progresso**: Sistema de log impede redownload de vídeos já processados
- **Gerenciamento de erros**: Captura e registra falhas no download

In [14]:
# Configurações
# audio_folder = "G:/Meu Drive/trabalho/audios/"
audio_folder = os.path.join(project_base_path, 'audios')
# log_file = "G:/Meu Drive/trabalho/download_log/download_progress.log"
log_file = os.path.join(project_base_path, 'download_log', 'download_progress.log')
# csv_file = "G:/Meu Drive/trabalho/filtrated_data_log/valid_videos.csv"
csv_file = os.path.join(project_base_path, 'filtrated_data_log', 'valid_videos.csv')

In [15]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [16]:
df = pd.read_csv(csv_file)
video_ids = df["video_id"].astype(str).tolist()

In [17]:
completed_videos = load_progress()

In [18]:
cookies = 'cookies_1306_02.txt'

for video_id in tqdm(video_ids, desc="Baixando áudios"):
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")

    if video_id in completed_videos or os.path.exists(audio_path):
        continue

    url = f"https://www.youtube.com/watch?v={video_id}"
    print(f"Baixando áudio de {url}...")

    try:
        command = [
            "yt-dlp",
            "--cookies", cookies,
            "-f", "bestaudio",
            "--extract-audio",
            "--audio-format", "mp3",
            "-o", f"{audio_folder}/%(id)s.%(ext)s",
            url
        ]
        
        result = subprocess.run(
            command,
            check=True, text=True, capture_output=True
        )

        print(f"Download concluído: {audio_path}")
        save_progress(video_id)

    except subprocess.CalledProcessError as e:
        print(f"Erro ao baixar {video_id}: {e.stderr}")

print("Processo de download concluído!")

Baixando áudios:   0%|          | 0/8019 [00:00<?, ?it/s]

Baixando áudio de https://www.youtube.com/watch?v=cPDsSCl4q2E...
Erro ao baixar cPDsSCl4q2E: ERROR: [youtube] cPDsSCl4q2E: Video unavailable. This video has been removed by the uploader

Baixando áudio de https://www.youtube.com/watch?v=sZIARxMPrMA...
Erro ao baixar sZIARxMPrMA: ERROR: [youtube] sZIARxMPrMA: Join this channel to get access to members-only content like this video, and other exclusive perks.

Baixando áudio de https://www.youtube.com/watch?v=B-mgjmPY8mY...
Erro ao baixar B-mgjmPY8mY: ERROR: [youtube] B-mgjmPY8mY: Join this channel to get access to members-only content like this video, and other exclusive perks.

Baixando áudio de https://www.youtube.com/watch?v=VuBCfqLdBgc...
Erro ao baixar VuBCfqLdBgc: ERROR: [youtube] VuBCfqLdBgc: Video unavailable. This video is private

Baixando áudio de https://www.youtube.com/watch?v=9MH_nRC6EFA...
Erro ao baixar 9MH_nRC6EFA: ERROR: [youtube] 9MH_nRC6EFA: Join this channel to get access to members-only content like this video, and 

---
## 4. Transcrição de Áudios

Transcreve os áudios baixados usando o modelo **Whisper** (OpenAI):
- **Modelo**: Small (executado em GPU via CUDA para melhor performance)
- **Idioma**: Português
- **Output**: Arquivos de texto (.txt) com transcrição completa
- **Otimização**: Após transcrição, o áudio é compactado (.zip) e o arquivo original é removido para economizar espaço
- **Sistema de checkpoints**: Evita retranscrever vídeos já processados

In [19]:
# audio_folder = "G:/Meu Drive/trabalho/audios/"
audio_folder = os.path.join(project_base_path, 'audios')
# transcription_folder = "G:/Meu Drive/trabalho/transcriptions"
transcription_folder = os.path.join(project_base_path, 'transcriptions')
# log_file = "G:/Meu Drive/trabalho/download_log/progresso_transcricao.log"
log_file = os.path.join(project_base_path, 'download_log', 'progresso_transcricao.log')

In [20]:
# df = pd.read_csv('G:/Meu Drive/trabalho/cleaned_data/final_comments_with_sentiment.csv')
df = pd.read_csv(os.path.join(project_base_path, 'cleaned_data', 'final_comments_with_sentiment.csv'))
# df = pd.read_csv(os.path.join(project_base_path, 'gabriel_video_ids.csv'))
# video_ids = df["video_id"].drop_duplicates().tolist()
# video_ids_a_processar = [vid for vid in video_ids if vid not in completed_videos]

In [21]:
df.shape

(494516, 20)

In [22]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [28]:
completed_videos = load_progress()

In [29]:
modelo = whisper.load_model("small").to("cuda")

In [30]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Transcrevendo áudios"):
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")
    output_text = os.path.join(transcription_folder, f"{video_id}.txt")
    zip_audio_base = os.path.join(audio_folder, video_id)  # Sem extensão

    # Se já foi transcrito, pula
    if video_id in completed_videos or os.path.exists(output_text):
        print(f"Transcrição já feita para {video_id}")
        continue

    if not os.path.exists(audio_path):
        print(f"Áudio não encontrado para {video_id}, pulando...")
        continue

    print(f"Transcrevendo {audio_path}...")

    try:
        result = modelo.transcribe(audio_path, language="portuguese")

        with open(output_text, "w", encoding="utf-8") as f:
            f.write(result["text"])

        save_progress(video_id)
        print(f"Transcrição salva: {output_text}")

        # Compacta apenas o arquivo MP3
        shutil.make_archive(zip_audio_base, 'zip', audio_folder, f"{video_id}.mp3")
        print(f"Áudio compactado: {zip_audio_base}.zip")
        
        os.remove(audio_path)
        print(f"Áudio excluído: {audio_path}")

    except Exception as e:
        print(f"Erro ao transcrever {video_id}: {str(e)}")

print("Processo de transcrição concluído!")

Transcrevendo áudios:   0%|          | 0/8155 [00:00<?, ?it/s]

Transcrição já feita para g3YDGLa2pWg
Transcrição já feita para BMIQPwdZvN0
Transcrição já feita para yvzlKwX8IFI
Áudio não encontrado para 86DW4tTw2BQ, pulando...
Transcrição já feita para grb8otb7UpA
Transcrição já feita para UY9wG7vcBpk
Transcrição já feita para CdIYBMmZZYs
Transcrição já feita para -ai_NXJ_R8s
Transcrição já feita para XmZ1egdsD9I
Transcrição já feita para ymRTffCnvWk
Transcrição já feita para s3LduqdkEcI
Transcrição já feita para gYtuPtVI_j8
Transcrição já feita para ouyoieJF37s
Transcrição já feita para RTlsGclr0KQ
Transcrição já feita para GVDhrTHKXDM
Transcrição já feita para M9R_Y5QSzHA
Transcrição já feita para tVBly-Rbdjg
Transcrição já feita para q3T3rqe6EO8
Transcrição já feita para WBzcydbJ8U4
Transcrição já feita para OpTAHtZVNBo
Transcrição já feita para M75oGrA2YjU
Transcrição já feita para tVOQboDOevE
Transcrição já feita para Jyro13oH-xM
Transcrição já feita para 1h2nFhbU528
Áudio não encontrado para m4v0_gYODoU, pulando...
Transcrição já feita para 